# Kalshi Market Data Demo

Demonstrates fetching open markets and orderbooks using the `kalshi_client` module.

## Setup

1. `pip install kalshi-python python-dotenv pandas`
2. Copy `.env.example` to `.env` and fill in your API credentials
3. Set `KALSHI_ENV=demo` for the demo exchange or `KALSHI_ENV=production` for live

**Note**: Uses the legacy `kalshi-python` SDK (v2.x) instead of `kalshi_python_sync` (v3.x) due to validation issues in the newer version.

In [ ]:
# Install dependencies (uncomment in Colab)
# !pip install kalshi-python python-dotenv pandas

In [1]:
'hi'

'hi'

In [1]:
from kalshi_client import KalshiMarketClient

# Reads KALSHI_ENV, KALSHI_API_KEY_ID, KALSHI_PRIVATE_KEY_PATH from .env
client = KalshiMarketClient()
print(f"Connected to Kalshi ({client.env})")

Connected to Kalshi (production)


## 1. Fetch open markets (first page)

In [13]:
df = client.get_open_markets(limit=10000)
print(f"{len(df)} markets returned")
df[[ "title", "status", "yes_ask", "no_ask"]].sample(3)

10000 markets returned


,title,status,yes_ask,no_ask
2723,"yes Kevin Durant: 15+,yes Pascal Siakam: 20+,y...",active,0,100
6572,"yes Dyson Daniels: 15+,yes Jalen Johnson: 20+,...",active,100,100
285,"yes Cleveland,yes Boston,yes Denver,yes San An...",active,12,100


In [9]:
df.columns

Index(['ticker', 'event_ticker', 'market_type', 'title', 'subtitle',
       'yes_sub_title', 'no_sub_title', 'created_time', 'updated_time',
       'open_time', 'close_time', 'expected_expiration_time',
       'expiration_time', 'latest_expiration_time', 'settlement_timer_seconds',
       'status', 'response_price_units', 'yes_bid', 'yes_bid_dollars',
       'yes_ask', 'yes_ask_dollars', 'no_bid', 'no_bid_dollars', 'no_ask',
       'no_ask_dollars', 'last_price', 'last_price_dollars', 'volume',
       'volume_fp', 'volume_24h', 'volume_24h_fp', 'result', 'can_close_early',
       'fractional_trading_enabled', 'open_interest', 'open_interest_fp',
       'notional_value', 'notional_value_dollars', 'previous_yes_bid',
       'previous_yes_bid_dollars', 'previous_yes_ask',
       'previous_yes_ask_dollars', 'previous_price', 'previous_price_dollars',
       'liquidity', 'liquidity_dollars', 'expiration_value', 'tick_size',
       'strike_type', 'custom_strike', 'rules_primary', 'rules_seco

In [21]:
df['status'].value_counts()

status
active    10000
Name: count, dtype: int64

In [20]:
df['yes_no_spread'] = df['yes_ask'] - df['no_ask']
df[['yes_no_spread', 'volume']].describe()

,yes_no_spread,volume
count,10000.000000,10000.000000
mean,-64.382000,50.199200
std,46.298255,764.897019
min,-100.000000,0.000000
25%,-100.000000,0.000000
50%,-100.000000,0.000000
75%,0.000000,0.000000
max,50.000000,45454.000000


## 2. Filter by event ticker

Pass `event_ticker` to narrow results to a specific event.

In [ ]:
# Example: filter by a specific event ticker
# Replace with a real event ticker from the DataFrame above
sample_event = df["event_ticker"].dropna().iloc[0] if "event_ticker" in df.columns and not df["event_ticker"].dropna().empty else None
if sample_event:
    event_df = client.get_open_markets(event_ticker=sample_event)
    print(f"Markets for event '{sample_event}': {len(event_df)}")
    event_df[["ticker", "title"]].head()

## 3. Filter by series ticker

In [ ]:
# Example: filter by series ticker
# Common series: "KXBTC" (Bitcoin), "KXINX" (S&P 500), etc.
# btc_df = client.get_open_markets(series_ticker="KXBTC")
# btc_df[["ticker", "title"]].head()

## 4. Search markets by keyword

In [ ]:
# Case-insensitive title search across open markets
results = client.search_markets("Bitcoin", limit=200)
print(f"Found {len(results)} markets matching 'Bitcoin'")
results[["ticker", "title"]].head(10)

## 5. Filter by close timestamp

Use `min_close_ts` / `max_close_ts` (Unix timestamps) to find markets
closing within a time window.

In [ ]:
import time

# Markets closing in the next 24 hours
now = int(time.time())
soon_df = client.get_open_markets(
    min_close_ts=now,
    max_close_ts=now + 86400,
    max_pages=1,
)
print(f"{len(soon_df)} markets closing in the next 24h")
soon_df[["ticker", "title"]].head(10)

## 6. Orderbook (depth=10 default)

In [ ]:
if not df.empty:
    sample_ticker = df["ticker"].iloc[0]
    ob = client.get_orderbook(sample_ticker)
    print(f"Orderbook for {ob['ticker']}")
    print("\nYES bids:")
    display(ob["yes"])
    print("\nNO bids:")
    display(ob["no"])

## 7. Single market detail

In [ ]:
if not df.empty:
    detail = client.get_market(df["ticker"].iloc[0])
    detail